In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re


In [7]:
import nltk
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to C:\Users\Lokesh
[nltk_data]     kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
# Load the dataset
train_df = pd.read_csv('data/train.csv')


In [9]:
import pandas as pd

# Load the test dataset (adjust the path as needed)
test_df = pd.read_csv('data/test.csv')

# For instance, if you need only certain columns, extract them here
test_texts = test_df['crimeaditionalinfo']  # Update based on actual column name in test dataset


In [17]:
# Check missing values in 'sub_category' and decide on handling them
train_df = train_df.dropna(subset=['sub_category'])

# Fill missing values in 'crimeaditionalinfo' with an empty string
train_df['crimeaditionalinfo'] = train_df['crimeaditionalinfo'].fillna('')
print("Remaining missing values:\n", train_df.isnull().sum())


Remaining missing values:
 category              0
sub_category          0
crimeaditionalinfo    0
dtype: int64


In [7]:
train_df = train_df.dropna(subset=['sub_category'])

# Fill missing values in crimeaditionalinfo with an empty string
train_df['crimeaditionalinfo'] = train_df['crimeaditionalinfo'].fillna('')
print("Missing values handled.")

# Balance the Dataset
from imblearn.over_sampling import RandomOverSampler

X = train_df.drop(columns=['sub_category', 'crimeaditionalinfo'])
y = train_df['sub_category']

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)
train_balanced = X_resampled.assign(sub_category=y_resampled)
print("Dataset balanced with oversampling.")


Missing values handled.
Dataset balanced with oversampling.


In [18]:
# Text Preprocessing
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Define Hinglish stopwords and set up stemmer
hinglish_stopwords = set(stopwords.words('english')).union({'hai', 'ka', 'ke', 'ki', 'ko', 'mein', 'se', 'ho'})
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = [stemmer.stem(word) for word in text.split() if word not in hinglish_stopwords]
    return ' '.join(words)

train_df['processed_text'] = train_df['crimeaditionalinfo'].apply(preprocess_text)
print("Text preprocessing complete.")

Text preprocessing complete.


In [19]:
# Feature Engineering
train_df['word_count'] = train_df['processed_text'].apply(lambda x: len(str(x).split()))
train_df['unique_word_ratio'] = train_df['processed_text'].apply(lambda x: len(set(str(x).split())) / len(str(x).split()) if x else 0)
train_df['average_word_length'] = train_df['processed_text'].apply(lambda x: sum(len(word) for word in str(x).split()) / len(str(x).split()) if x else 0)
print("Feature engineering complete.")

Feature engineering complete.


In [ ]:
from sklearn.model_selection import train_test_split

# Define features and target
X = train_df.drop(columns=['sub_category', 'crimeaditionalinfo'])
y = train_df['sub_category']

# Split the dataset without stratification
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print("Dataset split into training and validation sets without stratification.")


from sklearn.preprocessing import LabelEncoder
import joblib

# Combine `y_train` and `y_val` to ensure all labels are included in the encoding
all_labels = list(y_train) + list(y_val)

# Initialize and fit the encoder on the combined labels
label_encoder = LabelEncoder()
label_encoder.fit(all_labels)

# Encode `y_train` and `y_val` separately
y_train_encoded = label_encoder.transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

# Save the encoded labels and the label encoder for later use
joblib.dump(y_train_encoded, 'data/y_train_encoded.pkl')
joblib.dump(y_val_encoded, 'data/y_val_encoded.pkl')
joblib.dump(label_encoder, 'data/label_encoder.pkl')

print("Label encoding complete.")


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

# Balance the Dataset
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(train_df[['processed_text']], train_df['sub_category'])
print("Dataset balanced with oversampling.")

# Split the balanced data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)
print("Balanced dataset split into training and validation sets.")

# Encode labels
label_encoder = LabelEncoder()
all_labels = list(y_train) + list(y_val)
label_encoder.fit(all_labels)

y_train_encoded = label_encoder.transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

# Save the processed data and labels
joblib.dump(X_train, 'data/X_train.pkl')
joblib.dump(X_val, 'data/X_val.pkl')
joblib.dump(y_train_encoded, 'data/y_train_encoded.pkl')
joblib.dump(y_val_encoded, 'data/y_val_encoded.pkl')
joblib.dump(label_encoder, 'data/label_encoder.pkl')

print("Data preprocessing and encoding complete.")


Dataset balanced with oversampling.
Balanced dataset split into training and validation sets.
Data preprocessing and encoding complete.


In [7]:
# Data Preprocessing for Cyberfraud Classifier

# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import joblib

# Text preprocessing setup
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Acknowledgment: Define text preprocessing to clean and normalize Hinglish data
stop_words = set(stopwords.words('english')).union({'hai', 'ka', 'ke', 'ki', 'ko', 'mein', 'se', 'ho'})
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = [stemmer.stem(word) for word in text.split() if word not in stop_words]
    return ' '.join(words)

# Load and preprocess training data
train_df = pd.read_csv('data/train.csv')
train_df = train_df.dropna(subset=['sub_category'])
train_df['crimeaditionalinfo'] = train_df['crimeaditionalinfo'].fillna('')
train_df['processed_text'] = train_df['crimeaditionalinfo'].apply(preprocess_text)

# Encode training labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(train_df['sub_category'])

# Save processed training data
X_train = train_df[['processed_text']]
y_train = y_train_encoded
joblib.dump(X_train, 'data/X_train.pkl')
joblib.dump(y_train, 'data/y_train_encoded.pkl')

# --- Test Data Preparation ---

# Load and preprocess test data
test_df = pd.read_csv('data/test.csv')
test_df['crimeaditionalinfo'] = test_df['crimeaditionalinfo'].fillna('')
test_df['processed_text'] = test_df['crimeaditionalinfo'].apply(preprocess_text)

# Filter out unseen labels in the test set
# Retain only rows where 'sub_category' is in the training classes
test_df = test_df[test_df['sub_category'].isin(label_encoder.classes_)]

# Encode test labels if available
if 'sub_category' in test_df.columns:
    y_test_encoded = label_encoder.transform(test_df['sub_category'])

# Save processed test data
X_test = test_df[['processed_text']]
joblib.dump(X_test, 'data/X_test.pkl')
joblib.dump(y_test_encoded, 'data/y_test_encoded.pkl')

print("Preprocessing complete, processed data saved.")


Preprocessing complete, processed data saved.


In [5]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder
import re

# Define text preprocessing function
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

# Load training data
train_df = pd.read_csv('data/train.csv')
train_df['crimeaditionalinfo'] = train_df['crimeaditionalinfo'].fillna('')
train_df['processed_text'] = train_df['crimeaditionalinfo'].apply(preprocess_text)

# Encode training labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(train_df['sub_category'])

# Save processed training data
X_train = train_df[['processed_text']]
joblib.dump(X_train, 'data/X_train.pkl')
joblib.dump(y_train_encoded, 'data/y_train_encoded.pkl')

# --- Test Data Preparation ---
# Load test data
test_df = pd.read_csv('data/test.csv')
test_df['crimeaditionalinfo'] = test_df['crimeaditionalinfo'].fillna('')
test_df['processed_text'] = test_df['crimeaditionalinfo'].apply(preprocess_text)

# Handle unknown labels in test set
test_df['sub_category'] = test_df['sub_category'].map(
    lambda x: x if x in label_encoder.classes_ else train_df['sub_category'].mode()[0]
)

# Encode test labels
y_test_encoded = label_encoder.transform(test_df['sub_category'])

# Save processed test data
X_test = test_df[['processed_text']]
joblib.dump(X_test, 'data/X_test.pkl')
joblib.dump(y_test_encoded, 'data/y_test_encoded.pkl')

print("Preprocessing complete, processed data saved.")

Preprocessing complete, processed data saved.


In [12]:
# Keep your existing imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import joblib
import re

# Add these NLTK imports and downloads
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Initialize stemmer and stop words
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

# Your text preprocessing function
def preprocess_text(text):
    text = str(text).lower()  # Convert to string and lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters
    words = [stemmer.stem(word) for word in text.split() if word not in stop_words]
    return ' '.join(words)

# Load training data
train_df = pd.read_csv('data/train.csv')
train_df = train_df.dropna(subset=['sub_category'])
train_df['crimeaditionalinfo'] = train_df['crimeaditionalinfo'].fillna('')
train_df['processed_text'] = train_df['crimeaditionalinfo'].apply(preprocess_text)

# Initialize and fit label encoder on training data only
label_encoder = LabelEncoder()
label_encoder.fit(train_df['sub_category'])
joblib.dump(label_encoder, 'data/label_encoder.pkl')

# Process training data
y_train_encoded = label_encoder.transform(train_df['sub_category'])
X_train = train_df[['processed_text']]

# Save training data
joblib.dump(X_train, 'data/X_train.pkl')
joblib.dump(y_train_encoded, 'data/y_train_encoded.pkl')

# Process test data
test_df = pd.read_csv('data/test.csv')
test_df['crimeaditionalinfo'] = test_df['crimeaditionalinfo'].fillna('')
test_df['processed_text'] = test_df['crimeaditionalinfo'].apply(preprocess_text)

# Filter test data to only include labels seen during training
test_df = test_df[test_df['sub_category'].isin(label_encoder.classes_)]
y_test_encoded = label_encoder.transform(test_df['sub_category'])

# Save test data
X_test = test_df[['processed_text']]
joblib.dump(X_test, 'data/X_test.pkl')
joblib.dump(y_test_encoded, 'data/y_test_encoded.pkl')

# Print some information for verification
print("Number of training samples:", len(train_df))
print("Number of test samples:", len(test_df))
print("Number of unique labels:", len(label_encoder.classes_))
print("Preprocessing complete, processed data saved.")

[nltk_data] Downloading package stopwords to C:\Users\Lokesh
[nltk_data]     kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Number of training samples: 87095
Number of test samples: 28989
Number of unique labels: 35
Preprocessing complete, processed data saved.
